# Controlled benchmarks and optional CUDA smoke
Purpose: reusable mixer, encoder-block and whole-JEPA timing with identical token/mask workloads. Inputs: explicit grids, device and warm-up/repeat counts. Outputs: raw JSON-safe timing rows. Definitions only; import does not benchmark. Runtime: sweep typically 1–3 minutes. No checkpoints needed; rerun a sweep into a new artifact run directory.

Each case records forward and forward/backward separately, median and median absolute deviation, every timing sample, observed SDPA dispatcher events and actual parameter counts. Width equality is not parameter matching. A CPU peak-memory result is **not measured**, and is never replaced with a misleading RSS peak. CUDA measurement records baseline, total peak live allocated memory and incremental peak after separate warm-up; allocator caches are cleared between cases. Other system users/processes are not controlled. Full-JEPA timing includes predictor and teacher, excludes optimizer/EMA, and is explicitly separate from the mixer/block. No asymptotic fit is inferred from a short sweep.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
import gc, json, statistics, time
from pathlib import Path
import torch
from fmm_jepa.models import AttentionMixer, EncoderBlock, JEPA, observed_attention_backend
from fmm_jepa.operators import FMMMixer
from fmm_jepa.geometry import clear_geometry_cache
from fmm_jepa.data import sample_masks, synthetic_images
from fmm_jepa.training import move_masks

def _sync(device):
    if str(device).startswith('cuda'): torch.cuda.synchronize(device)

def _profile_backend(call, device):
    with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as trace:
        with torch.no_grad(): call()
        _sync(device)
    keys=sorted({event.key for event in trace.key_averages() if 'scaled_dot_product' in event.key})
    if any('flash_attention_for_cpu' in key for key in keys): backend='torch SDPA CPU flash'
    elif any('flash_attention' in key for key in keys): backend='torch SDPA CUDA flash'
    elif any('efficient_attention' in key for key in keys): backend='torch SDPA memory-efficient'
    elif any('cudnn_attention' in key for key in keys): backend='torch SDPA cuDNN'
    elif any('attention_math' in key for key in keys): backend='torch SDPA math'
    else: backend='no SDPA operator observed' if not keys else 'torch SDPA unclassified'
    return backend, keys

def benchmark_case(kind, grid, scope, device='cpu', dtype=torch.float32, dim=32, heads=4,
                   order=2, leaf_size=2, batch_size=1, warmup=2, repeats=5, seed=191):
    if warmup < 1 or repeats < 3: raise ValueError('At least one warm-up and three measurements required')
    clear_geometry_cache(); gc.collect()
    if str(device).startswith('cuda'): torch.cuda.empty_cache()
    torch.manual_seed(seed)
    rows,cols=map(int,grid); tokens=rows*cols
    indices=valid=x=None
    if scope != "full_jepa":
        indices=torch.arange(tokens,device=device).expand(batch_size,-1)
        valid=torch.ones_like(indices,dtype=torch.bool)
        x=torch.randn(batch_size,tokens,dim,device=device,dtype=dtype,requires_grad=True)
    if scope=='mixer':
        model=(FMMMixer(dim,order,leaf_size) if kind=='fmm' else AttentionMixer(dim,heads)).to(device=device,dtype=dtype)
        def call(): return model(x,indices,valid,(rows,cols))
        visible=tokens; workload='all original patches; no visibility reduction'
    elif scope=='encoder_block':
        model=EncoderBlock(kind,dim,heads,order,leaf_size).to(device=device,dtype=dtype)
        def call(): return model(x,indices,valid,(rows,cols))
        visible=tokens; workload='all original patches; identical pre-LN/residual/MLP scaffolding'
    elif scope=='full_jepa':
        if dtype!=torch.float32: raise ValueError('Full-JEPA benchmark restricted to float32')
        model=JEPA({'kind':kind,'patch_size':4,'dim':dim,'depth':1,'heads':heads,'order':order,'leaf_size':leaf_size,
                    'receiver_gate':True,'predictor_dim':dim,'predictor_depth':1,'predictor_heads':heads}).to(device)
        generator=torch.Generator().manual_seed(seed)
        images=synthetic_images(batch_size,rows*4,cols*4,generator).to(device)
        masks=move_masks(sample_masks((rows,cols),batch_size,generator=generator),device)
        def call(): return model(images,masks['context_indices'],masks['context_valid'],masks['target_indices'],masks['target_valid'])['loss']
        visible=float(masks['context_valid'].sum(1).float().mean())
        workload='identical seeded context and target masks; full-grid teacher; forward/backward excludes optimizer and EMA'
    else: raise ValueError(scope)
    model.eval()
    backend,events=_profile_backend(call,device)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    # Initialize both autograd paths and lazy device workspaces before any timed or memory row.
    with torch.no_grad(): precondition=call()
    del precondition
    precondition=call()
    (precondition if precondition.ndim==0 else precondition.square().mean()).backward()
    del precondition
    model.zero_grad(set_to_none=True)
    if x is not None: x.grad=None
    _sync(device)
    result=[]
    for mode in ('forward','forward_backward'):
        def iteration():
            model.zero_grad(set_to_none=True)
            if x is not None and x.grad is not None: x.grad=None
            if mode=='forward':
                with torch.no_grad(): output=call()
            else:
                output=call()
                scalar=output if output.ndim==0 else output.square().mean()
                scalar.backward()
            return output
        for _ in range(warmup): output=iteration()
        del output
        _sync(device)
        values=[]
        for _ in range(repeats):
            _sync(device); start=time.perf_counter(); output=iteration();_sync(device)
            values.append((time.perf_counter()-start)*1000)
            del output
        # Separate allocator measurement from timings. CUDA live allocations only, cached reservations excluded.
        model.zero_grad(set_to_none=True)
        if x is not None: x.grad=None
        gc.collect();_sync(device)
        if str(device).startswith('cuda'):
            torch.cuda.empty_cache();base=torch.cuda.memory_allocated(device)
            torch.cuda.reset_peak_memory_stats(device);output=iteration();_sync(device)
            peak=torch.cuda.max_memory_allocated(device);increment=peak-base;del output
            memory='CUDA max_memory_allocated: total includes parameters, inputs, gradients and other live allocations; incremental=peak-baseline; excludes cached reservations'
        else:
            base=peak=increment=None
            memory='CPU peak tensor memory NOT_RUN; process RSS is not a comparable allocator peak and is not substituted'
        median=statistics.median(values)
        result.append({'kind':kind,'scope':scope,'mode':mode,'rows':rows,'cols':cols,'N_grid':tokens,'N_visible_mean':visible,
                       'dim':dim,'order':order if kind=='fmm' else None,'batch_size':batch_size,'device':str(device),'dtype':str(dtype),
                       'warmup':warmup,'repeats':repeats,'median_ms':median,'mad_ms':statistics.median(abs(v-median) for v in values),
                       'min_ms':min(values),'max_ms':max(values),'raw_ms':json.dumps(values),'backend':backend,'backend_events':json.dumps(events),
                       'trainable_parameters':trainable,'baseline_allocated_bytes':base,'peak_allocated_bytes':peak,
                       'incremental_peak_bytes':increment,'memory_definition':memory,'workload':workload,'status':'PASS'})
    del model,x,indices,valid,call
    clear_geometry_cache(); gc.collect()
    if str(device).startswith('cuda'): torch.cuda.empty_cache()
    return result

def benchmark_sweep(grids, device='cpu', scopes=('mixer','encoder_block','full_jepa'), **kwargs):
    result=[]
    for grid in grids:
        for scope in scopes:
            for kind in ('fmm','vit'):
                result.extend(benchmark_case(kind,grid,scope,device=device,**kwargs))
    return result

def cuda_smoke_test(config):
    if not torch.cuda.is_available(): return {'optional_cuda':{'status':'NOT_RUN','details':'No CUDA device available; CPU correctness remains required'}}
    from fmm_jepa.training import make_training_state, synthetic_step
    import copy
    cfg=copy.deepcopy(config);cfg.update(device='cuda:0',image_height=32,image_width=48,batch_size=1)
    cfg['model'].update(dim=16,depth=1,predictor_dim=16)
    try:
        state=make_training_state(cfg); row=synthetic_step(*state,cfg,1);torch.cuda.synchronize()
        assert all(p.grad is None for p in state[0].target.parameters())
        report={'status':'PASS','details':{'device':torch.cuda.get_device_name(0),'dtype':'float32','grid':[cfg['image_height']//cfg['model']['patch_size'],cfg['image_width']//cfg['model']['patch_size']],'loss':row['loss'],'finite_forward_backward_ema':True,'exact_resume':'NOT_RUN on CUDA'}}
        del state
    except Exception as error:
        report={'status':'FAIL','details':f'{type(error).__name__}: {error}'}
    gc.collect();torch.cuda.empty_cache()
    return {'optional_cuda':report}
